In [1]:
import pandas as pd
import numpy as np

import cv2
from pathlib import Path
from typing import Optional, Any
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

cv2.setNumThreads(0)

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
PROCESED_DATA = DATA / 'processed'
MEL_SPECTROGRAMS = PROCESED_DATA / 'mel_spectrograms'
BRONCHITIS = MEL_SPECTROGRAMS / 'bronchitis'
COVID = MEL_SPECTROGRAMS / 'covid'
HEALTHY = MEL_SPECTROGRAMS / 'healthy'
PNEUMONIA = MEL_SPECTROGRAMS / 'pneumonia'
TUBERCULOSIS = MEL_SPECTROGRAMS / 'tuberculosis'

VALID_LABELS = {"bronchitis", "covid", "healthy", "pneumonia", "tuberculosis"}


In [3]:
def build_image_manifest(
    spectrogram_root: Path,
    include_nan_folder: bool = False,
) -> pd.DataFrame:
    """Scan mel_spectrograms/<label>/*.png into a manifest dataframe."""
    records = []
    for label_dir in spectrogram_root.iterdir():
        if not label_dir.is_dir():
            continue

        label = label_dir.name.lower()
        if label == "nan" and not include_nan_folder:
            continue
        if label != "nan" and label not in VALID_LABELS:
            print(f"Warning, unexpected folder found: {label_dir.name}")

        for image_path in label_dir.glob("*.png"):
            records.append({
                "image_path": str(image_path),
                "label": label,
                "sample_id": image_path.stem,
            })

    manifest = pd.DataFrame(records)
    print(f"Found {len(manifest)} images across {manifest['label'].nunique()} labels")
    print(manifest["label"].value_counts())
    return manifest

In [4]:
def build_fusion_manifest(
    features_csv_path: Path,
    spectrogram_root: Path,
    audio_filename_column: str,
) -> pd.DataFrame:
    """Join tabular features with image paths by matching filename stems."""

    features_df = pd.read_csv(features_csv_path)
    features_df["sample_id"] = features_df[audio_filename_column].apply(
        lambda name: Path(name).stem
    )

    image_manifest = build_image_manifest(spectrogram_root)

    # avoid duplicate "label" column collision, keep only image_path + sample_id from image_manifest
    image_manifest_slim = image_manifest[["image_path", "sample_id"]]

    merged = features_df.merge(
        image_manifest_slim,
        on="sample_id",
        how="inner",
    )

    dropped = len(features_df) - len(merged)
    if dropped > 0:
        unmatched = set(features_df["sample_id"]) - set(image_manifest["sample_id"])
        print(f"Warning, {dropped} rows from features.csv had no matching image")
        print(f"Example unmatched IDs: {list(unmatched)[:5]}")

    matched_pct = len(merged) / len(features_df) * 100
    print(f"Matched {len(merged)}/{len(features_df)} rows ({matched_pct:.1f}%)")

    return merged

In [5]:
def create_stratified_splits(
    manifest: pd.DataFrame,
    label_column: str = "label",
    train_size: float = 0.70,
    val_size: float = 0.15,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split manifest into train/val/test, preserving class proportions."""

    train_df, temp_df = train_test_split(
        manifest,
        train_size=train_size,
        stratify=manifest[label_column],
        random_state=random_state,
    )

    relative_val_size = val_size / (1 - train_size)
    val_df, test_df = train_test_split(
        temp_df,
        train_size=relative_val_size,
        stratify=temp_df[label_column],
        random_state=random_state,
    )

    for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"\n{name} split, {len(split_df)} samples")
        print(split_df[label_column].value_counts())

    return train_df, val_df, test_df

In [6]:
LABEL_TO_IDX = {
    "healthy": 0,
    "covid": 1,
    "tuberculosis": 2,
    "bronchitis": 3,
    "pneumonia": 4,
}


IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}


def compute_class_weights(train_df: pd.DataFrame, label_column: str = "label") -> torch.Tensor:
    """Inverse frequency weighting, rare classes get higher weight."""
    labels = train_df[label_column].map(LABEL_TO_IDX).values
    classes = np.arange(len(LABEL_TO_IDX))

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=labels,
    )
    return torch.tensor(weights, dtype=torch.float32)


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    running_loss = 0.0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


In [7]:
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(dataloader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    per_class_report = classification_report(
        all_labels,
        all_preds,
        target_names=[IDX_TO_LABEL[i] for i in range(len(LABEL_TO_IDX))],
        zero_division=0,
    )
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return {
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "report": per_class_report,
        "confusion_matrix": conf_matrix,
    }


In [8]:
def train_cnn(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    num_epochs: int = 30,
    batch_size: int = 32,
    learning_rate: float = 1e-3,
    checkpoint_dir: Path = Path("checkpoints"),
) -> nn.Module:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")

    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = SpectrogramDataset(train_df)
    val_dataset = SpectrogramDataset(val_df)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = SimpleCoughCNN(num_classes=len(LABEL_TO_IDX)).to(device)

    class_weights = compute_class_weights(train_df).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

    best_macro_f1 = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device)

        scheduler.step(val_metrics["macro_f1"])

        print(f"\nEpoch {epoch}/{num_epochs}")
        print(f"Train loss, {train_loss:.4f} | Val loss, {val_metrics['val_loss']:.4f} | Val macro F1, {val_metrics['macro_f1']:.4f}")

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), checkpoint_dir / "best_cnn_model.pt")
            print(f"New best model saved, macro F1 {best_macro_f1:.4f}")

    print("\nTraining complete, final validation report at best checkpoint")
    model.load_state_dict(torch.load(checkpoint_dir / "best_cnn_model.pt"))
    final_metrics = evaluate(model, val_loader, criterion, device)
    print(final_metrics["report"])
    print("Confusion matrix, rows true, cols predicted")
    print(final_metrics["confusion_matrix"])

    return model

In [9]:


# ---------------------------------------------------------------------------
# Labels
# ---------------------------------------------------------------------------
LABEL_TO_IDX = {
    "healthy": 0,
    "covid": 1,
    "tuberculosis": 2,
    "bronchitis": 3,
    "pneumonia": 4,
}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
class SpectrogramDataset(Dataset):
    """Phase 1, CNN only, loads spectrogram images from the manifest."""

    def __init__(
        self, manifest: pd.DataFrame, transform: Optional[Any] = None
    ) -> None:
        self.manifest = manifest.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        row = self.manifest.iloc[idx]
        image = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"Image not found at {row['image_path']}")

        image = image.astype(np.float32) / 255.0
        image_tensor = torch.from_numpy(image).unsqueeze(0)  # (1, H, W)

        if self.transform:
            image_tensor = self.transform(image_tensor)

        label = LABEL_TO_IDX[row["label"]]
        return image_tensor, label


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class SimpleCoughCNN(nn.Module):
    def __init__(self, num_classes: int = 5) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


# ---------------------------------------------------------------------------
# Class weights
# ---------------------------------------------------------------------------
def compute_class_weights(train_df: pd.DataFrame, label_column: str = "label") -> torch.Tensor:
    labels = train_df[label_column].map(LABEL_TO_IDX).values
    classes = np.arange(len(LABEL_TO_IDX))
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    return torch.tensor(weights, dtype=torch.float32)


# ---------------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------------

def train_one_epoch(model, dataloader, criterion, optimizer, device) -> float:
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(dataloader.dataset)


def evaluate(model, dataloader, criterion, device) -> dict:
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    progress_bar = tqdm(dataloader, desc="Validating", leave=False)
    with torch.no_grad():
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss = running_loss / len(dataloader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    report = classification_report(
        all_labels, all_preds,
        target_names=[IDX_TO_LABEL[i] for i in range(len(LABEL_TO_IDX))],
        zero_division=0,
    )
    conf_matrix = confusion_matrix(all_labels, all_preds)
    return {"val_loss": val_loss, "macro_f1": macro_f1, "report": report, "confusion_matrix": conf_matrix}


# ---------------------------------------------------------------------------
# Training entrypoint
# ---------------------------------------------------------------------------
def train_cnn(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    num_epochs: int = 30,
    batch_size: int = 32,
    learning_rate: float = 1e-3,
    checkpoint_dir: Path = Path("checkpoints"),
) -> nn.Module:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(SpectrogramDataset(train_df), batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(SpectrogramDataset(val_df), batch_size=batch_size, shuffle=False, num_workers=2)

    model = SimpleCoughCNN(num_classes=len(LABEL_TO_IDX)).to(device)
    class_weights = compute_class_weights(train_df).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

    best_macro_f1 = 0.0
    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Epochs")

    for epoch in epoch_bar:
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_metrics["macro_f1"])

        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            val_loss=f"{val_metrics['val_loss']:.4f}",
            val_f1=f"{val_metrics['macro_f1']:.4f}",
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), checkpoint_dir / "best_cnn_model.pt")
            tqdm.write(f"Epoch {epoch}, new best model saved, macro F1 {best_macro_f1:.4f}")

    model.load_state_dict(torch.load(checkpoint_dir / "best_cnn_model.pt"))
    final_metrics = evaluate(model, val_loader, criterion, device)
    print("\nFinal validation report at best checkpoint")
    print(final_metrics["report"])
    print("Confusion matrix, rows true, cols predicted")
    print(final_metrics["confusion_matrix"])

    return model

In [ ]:
if __name__ == "__main__":
    cv2.setNumThreads(0) 

    merged_manifest = build_fusion_manifest(
        features_csv_path=Path("../data/processed/combined_features.csv"),
        spectrogram_root=Path("../data/processed/mel_spectrograms"),
        audio_filename_column="filename",
    )

    train_df, val_df, test_df = create_stratified_splits(merged_manifest)    

    model = train_cnn(train_df, val_df, num_epochs=15, batch_size=32, learning_rate=1e-3)

Found 2860 images across 5 labels
label
healthy         1012
tuberculosis     840
covid            835
bronchitis        91
pneumonia         82
Name: count, dtype: int64
Warning, 8 rows from features.csv had no matching image
Example unmatched IDs: ['PID_200A_5_pixel_0', 'PID_200A_4_codec', 'PID_200A_1_codec_0', 'PID_200A_4_pixel', 'PID_200A_3_pixel']
Matched 2860/2868 rows (99.7%)

train split, 2001 samples
label
healthy         708
tuberculosis    588
covid           584
bronchitis       64
pneumonia        57
Name: count, dtype: int64

val split, 429 samples
label
healthy         152
tuberculosis    126
covid           125
bronchitis       13
pneumonia        13
Name: count, dtype: int64

test split, 430 samples
label
healthy         152
tuberculosis    126
covid           126
bronchitis       14
pneumonia        12
Name: count, dtype: int64
Training on cpu


Epochs:   0%|          | 0/15 [00:00<?, ?it/s]